In [ ]:
import os
import fitz  # PyMuPDF pour extraire le texte des PDF
import re
import nltk
import numpy as np
from nltk.corpus import stopwords
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# T?l?charger les stopwords en fran?ais
nltk.download('stopwords')
stop_words = set(stopwords.words('french'))

In [ ]:
# Fonction pour extraire le texte d'un PDF
def extract_text_from_pdf(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        text = " ".join([page.get_text("text") for page in doc])
        return text
    except Exception as e:
        print(f"Erreur de lecture {pdf_path}: {e}")
        return ""

In [ ]:
# Fonction de pr?traitement du texte
def preprocess_text(text):
    text = unidecode(text)  # Supprime les accents (ex: "d?veloppeur" -> "developpeur")
    text = re.sub(r'\s+', ' ', text)  # Supprime les espaces multiples
    text = re.sub(r'[^\w\s]', '', text)  # Supprime la ponctuation
    text = ' '.join([word.lower() for word in text.split() if word.lower() not in stop_words])
    return text


In [ ]:
# Dossier contenant les CVs (modifier selon votre chemin)
cv_folder = r"C:\Users\Asus\Desktop\donner\cv"

In [ ]:
# Description du poste en fran?ais
job_description = """
Nous recherchons un ing?nieur en informatique sp?cialis? en Python, Intelligence Artificielle et D?veloppement Web.
Exp?rience avec Flask, Angular et bases de donn?es est un plus.
"""  # Modifier selon le besoin

In [ ]:
# Lire et traiter les CVs
cv_files = [os.path.join(cv_folder, f) for f in os.listdir(cv_folder) if f.endswith('.pdf')]
cv_texts = [preprocess_text(extract_text_from_pdf(f)) for f in cv_files]


In [ ]:
# Ajouter la description du poste comme r?f?rence
documents = [preprocess_text(job_description)] + cv_texts

# Transformer les textes en vecteurs TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

In [ ]:
# Calculer la similarit? cosinus entre la description du poste et les CVs
cosine_similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

# Classer les CVs en fonction de leur pertinence
ranked_indices = np.argsort(cosine_similarities)[::-1]  # Tri d?croissant


In [ ]:
# Affichage des r?sultats
print("\n?? Classement des CVs selon la pertinence:")
for i, idx in enumerate(ranked_indices):
    print(f"Rang {i+1}: {cv_files[idx]} - Score: {cosine_similarities[idx]:.4f}")